In [13]:
import sys
import os
sys.path.append('../')
import pathlib as pl
from SymEigen import *
from sympy import symbols
from project_dir import backend_source_dir

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction("__host__ __device__")
Gen.DisableLatexComment()

In [14]:
def compute_x_tilde(x_n, x_n_1, v_n, v_n_1, h):
    return (4 * x_n - x_n_1 + 2 * h / 3 * (4 * v_n - v_n_1)) / 3

def compute_x_tilde_with_gravity(x_n, x_n_1, v_n, v_n_1, g, h):
    return compute_x_tilde(x_n, x_n_1, v_n, v_n_1, h) + 4 * (g * h * h)/9

def compute_v(x, x_n, x_n_1, h):
    return (3 * x - 4 * x_n + x_n_1) / (2 * h)

x = symbols('x')
x_n = symbols('x_n')
x_n_1 = symbols('x_{n-1}')
v_n = symbols('v_n')
v_n_1 = symbols('v_{n-1}')
h = symbols('h')
g = symbols('g')

x_tilde = compute_x_tilde(x_n, x_n_1, v_n, v_n_1, h)
display(x_tilde)

x_tilde_with_gravity = compute_x_tilde_with_gravity(x_n, x_n_1, v_n, v_n_1, g, h)
display(x_tilde_with_gravity)

v = compute_v(x, x_n, x_n_1, h)
display(v)

2*h*(4*v_n - v_{n-1})/9 + 4*x_n/3 - x_{n-1}/3

4*g*h**2/9 + 2*h*(4*v_n - v_{n-1})/9 + 4*x_n/3 - x_{n-1}/3

(3*x - 4*x_n + x_{n-1})/(2*h)

# Affine Body BDF2

In [15]:
x = Eigen.Vector('q', 12)
x_n = Eigen.Vector('q_n', 12)
x_n_1 = Eigen.Vector('q_n_1', 12)
v_n = Eigen.Vector('qv_n', 12)
v_n_1 = Eigen.Vector('qv_n_1', 12)
h = Eigen.Scalar('h')
g = Eigen.Vector('g', 12)
x_tilde = compute_x_tilde(x_n, x_n_1, v_n, v_n_1, h[0,0])
Cl0 = Gen.Closure(x_n, x_n_1, v_n, v_n_1, h)
x_tilde_with_gravity = compute_x_tilde_with_gravity(x_n, x_n_1, v_n, v_n_1, g, h[0,0])
Cl1 = Gen.Closure(x_n, x_n_1, v_n, v_n_1, g, h)
v = compute_v(x, x_n, x_n_1, h[0,0])
Cl2 = Gen.Closure(x, x_n, x_n_1, h)

abd_content =f''' // BDF2 Affine Body

{Cl0('compute_q_tilde', x_tilde)}

{Cl1('compute_q_tilde', x_tilde_with_gravity)}

{Cl2('compute_qv', v)}
'''

f = open(backend_source_dir('cuda') / 'affine_body/bdf/sym/bdf2.inl', 'w')
f.write(abd_content)
f.close()

# Finite Element BDF2

In [16]:
x = Eigen.Vector('x', 3)
x_n = Eigen.Vector('x_n', 3)
x_n_1 = Eigen.Vector('x_n_1', 3)
v_n = Eigen.Vector('v_n', 3)
v_n_1 = Eigen.Vector('v_n_1', 3)
h = Eigen.Scalar('h')
g = Eigen.Vector('g', 3)
x_tilde = compute_x_tilde(x_n, x_n_1, v_n, v_n_1, h[0,0])
Cl0 = Gen.Closure(x_n, x_n_1, v_n, v_n_1, h)
x_tilde_with_gravity = compute_x_tilde_with_gravity(x_n, x_n_1, v_n, v_n_1, g, h[0,0])
Cl1 = Gen.Closure(x_n, x_n_1, v_n, v_n_1, g, h)
v = compute_v(x, x_n, x_n_1, h[0,0])
Cl2 = Gen.Closure(x, x_n, x_n_1, h)

fem_content = f''' // BDF2 Finite Element

{Cl0('compute_x_tilde', x_tilde)}

{Cl1('compute_x_tilde', x_tilde_with_gravity)}

{Cl2('compute_v', v)}
'''

f = open(backend_source_dir('cuda') / 'finite_element/bdf/sym/bdf2.inl', 'w')
f.write(fem_content)
f.close()